In [3]:
import io, re, sys, json
import numpy as np, pandas as pd

In [8]:
# ---------- 1. Leitura ----------
# O arquivo vem em UTF-16 LE (sem BOM), separador ';', decimal ',' e milhar '.'
import urllib.request
url = "https://raw.githubusercontent.com/giordanobrandao/DataVisProjetoFinal/main/snis-raw-data.csv"
with urllib.request.urlopen(url) as resp:
    txt = resp.read().decode("utf-16-le")

bruto = pd.read_csv(io.StringIO(txt), sep=";", dtype=str, index_col=False)
bruto = bruto[bruto["Código do Município"] != "TOTAL da AMOSTRA:"].copy()

In [9]:
# renomeia colunas para o código curto (AG001, ES001, IN055, ...)
descr = {}
novas = []
for c in bruto.columns:
    m = re.match(r"^([A-Z]{2}\d{3}[A-Z]?|POP_TOT|POP_URB)\s*-\s*(.*)$", c)
    if m:
        novas.append(m.group(1)); descr[m.group(1)] = m.group(2).strip()
    else:
        novas.append(c)
bruto.columns = novas

NUM = re.compile(r"^-?\d{1,3}(\.\d{3})*(,\d+)?$|^-?\d+(,\d+)?$")
def num(s):
    if not isinstance(s, str): return np.nan
    s = s.strip()
    if s == "" or not NUM.match(s): return np.nan     # texto ("Sede Municipal") ou vazio -> NaN
    return float(s.replace(".", "").replace(",", "."))

In [10]:
# ---------- 2. Identificação ----------
def dv_ibge(c6):
    """Dígito verificador do código IBGE de município (pesos 1,2,1,2,1,2)."""
    soma = 0
    for i, ch in enumerate(f"{c6:06d}"):
        p = int(ch) * (1 if i % 2 == 0 else 2)
        soma += p // 10 + p % 10
    return (10 - soma % 10) % 10

cols = {}
cols["cod6"] = bruto["Código do Município"].astype(int)
cols["cod7"] = cols["cod6"].map(lambda c: int(f"{c:06d}{dv_ibge(c)}"))
assert dv_ibge(230440) == 0 and dv_ibge(231290) == 8       # Fortaleza 2304400, Sobral 2312908

nome = bruto["Município"].str.strip()
nome = nome.replace({"Acarapé": "Acarape"})               # mesmo município (230015), duas grafias
cols["municipio"] = nome
cols["ano"] = bruto["Ano de Referência"].astype(int)
cols["servico"] = bruto["Tipo de serviço"].str.strip()

for c in bruto.columns:
    if re.match(r"^(AG|ES|FN|IN|QD)\d", c) or c in ("POP_TOT", "POP_URB"):
        cols[c] = bruto[c].map(num)
d = pd.DataFrame(cols).sort_values(["cod7", "ano"]).reset_index(drop=True)
assert not d.duplicated(["cod7", "ano"]).any()

In [11]:
# ---------- 3. Indicadores derivados (passo 2) ----------
d["cob_agua"] = d.AG001 / d.POP_TOT * 100                 # % da população total
d["cob_esgoto"] = np.where(d.servico == "Água e Esgoto", d.ES001 / d.POP_TOT * 100, np.nan)
d["sem_agua"] = d.POP_TOT - d.AG001                       # pessoas sem atendimento (déficit absoluto)
d["sem_esgoto"] = np.where(d.servico == "Água e Esgoto", d.POP_TOT - d.ES001, np.nan)

# IPCA acumulado no ano (dez/dez), IBGE. 1995-2000, 2021 e 2022 conferidos em publicações do IBGE;
# 2001-2020: conferir com SIDRA (tabela 1737) antes da versão final.
IPCA = {1995:22.41,1996:9.56,1997:5.22,1998:1.65,1999:8.94,2000:5.97,2001:7.67,2002:12.53,2003:9.30,
        2004:7.60,2005:5.69,2006:3.14,2007:4.46,2008:5.90,2009:4.31,2010:5.91,2011:6.50,2012:5.84,
        2013:5.91,2014:6.41,2015:10.67,2016:6.29,2017:2.95,2018:3.75,2019:4.31,2020:4.52,2021:10.06,2022:5.79}
anos = sorted(IPCA)
indice = {}; acc = 1.0
for a in anos:
    acc *= 1 + IPCA[a] / 100; indice[a] = acc              # índice de preços com fim de cada ano
fator = {a: indice[2022] / indice[a] for a in anos}        # valores nominais do ano a -> R$ de dez/2022
d["fator_ipca"] = d.ano.map(fator)
d["inv_total_r22"] = d.FN033 * d.fator_ipca                # investimento total do prestador, R$ de 2022
d["inv_agua_r22"] = d.FN023 * d.fator_ipca
d["inv_esgoto_r22"] = d.FN024 * d.fator_ipca

In [12]:
# ---------- 4. Saídas ----------
import os
os.makedirs("dados", exist_ok=True)
d.to_csv("dados/snis_ce_limpo.csv", index=False, encoding="utf-8")

def grupo(c):
    return {"AG":"Água","ES":"Esgoto","FN":"Financeiro","IN":"Indicador calculado","QD":"Qualidade","GE":"Geral","PO":"População"}.get(c[:2], "Identificação")
dic = pd.DataFrame({"codigo": list(descr), "descricao": [descr[k] for k in descr]})
dic["grupo"] = dic.codigo.map(grupo)
dic["pct_preenchido"] = dic.codigo.map(lambda k: round(d[k].notna().mean()*100,1) if k in d else np.nan)
dic = dic[dic.codigo.isin(d.columns)]
dic.to_csv("dados/dicionario_variaveis.csv", index=False, encoding="utf-8")

In [13]:
# painel enxuto para o D3 (recorte analítico: 2001-2022; 1995-2000 têm poucos municípios e muitas lacunas)
def r(x, k=0):
    return None if pd.isna(x) else (int(round(x)) if k == 0 else round(float(x), k))
e = d[d.ano >= 2001]
mun = []
for cod7, g in e.groupby("cod7"):
    mun.append({
        "id": int(cod7), "nome": g.municipio.iloc[-1],
        "linhas": [[int(x.ano), 1 if x.servico == "Água e Esgoto" else 0, r(x.POP_TOT), r(x.AG001), r(x.ES001),
                    r(x.inv_total_r22), r(x.IN049, 1), r(x.IN004, 2)] for x in g.itertuples()]})
mun.sort(key=lambda m: m["nome"])
json.dump({"campos": ["ano", "svc", "pop", "agua", "esgoto", "inv_r22", "perdas", "tarifa"],
           "svc": "0 = só água; 1 = água e esgoto",
           "fonte": "SNIS - Série Histórica (CAGECE), CE; IPCA/IBGE para deflacionar",
           "municipios": mun}, open("dados/painel_saneamento.json", "w", encoding="utf-8"),
          ensure_ascii=False, separators=(",", ":"))
print("ok:", d.shape, "municípios:", d.cod7.nunique(), "| json municípios:", len(mun))

ok: (3743, 223) municípios: 164 | json municípios: 164
